---
# 🚇 Urban Growth Dynamics of ASEAN Megacities
## Through the Lens of Public Transport Ridership
### การวิเคราะห์การเติบโตของมหานครอาเซียนผ่านเลนส์ระบบขนส่งสาธารณะ
---
## 📋 Executive Summary — บทสรุปผู้บริหาร

&emsp;ลองนึกภาพตามนะคะ — ทุกเช้า คนหลายสิบล้านคนในอาเซียนออกจากบ้าน บางคนขึ้น BTS บางคนนั่ง MRT สิงคโปร์ บางคนต่อรถเมล์ในกัวลาลัมเปอร์ การเคลื่อนไหวของคนเหล่านี้ไม่ใช่แค่ "การเดินทาง" — มันคือ **สัญญาณชีพ** ของเมือง

&emsp;เมื่อเศรษฐกิจดี คนออกไปทำงาน ช้อปปิ้ง พบปะสังสรรค์ — ตัวเลขผู้โดยสารพุ่งขึ้น เมื่อเกิดวิกฤต เช่น COVID-19 — ตัวเลขดิ่งลงในชั่วข้ามคืน นี่คือสิ่งที่รายงานฉบับนี้ต้องการพิสูจน์

| ส่วน | เมือง/หัวข้อ | สิ่งที่จะค้นพบ |
|------|-------------|----------------|
| **Part 1** | กรุงเทพฯ 🇹🇭 | คนกรุงเทพฯ เดินทางด้วยอะไร เพิ่มขึ้นแค่ไหน? |
| **Part 2** | สิงคโปร์ 🇸🇬 | ระบบขนส่งระดับโลกเขาทำได้อย่างไร? |
| **Part 3** | กัวลาลัมเปอร์ 🇲🇾 | มาเลย์กำลังไปในทิศทางไหน? |
| **Part 4** | จาการ์ตา 🇮🇩 | เมืองที่ใหญ่ที่สุดในอาเซียนอยู่ตรงไหน? |
| **Part 5** | มะนิลา 🇵🇭 | ฟิลิปปินส์โตมา 25 ปีอย่างไร? |
| **Part 6** | Economic Growth | ขนส่งกับเศรษฐกิจ — มีความสัมพันธ์กันจริงไหม? |
| **Part 7** | Development Gap | เมืองกำลังพัฒนาตามสิงคโปร์ทัน? |

---

## 🗂️ ที่มาและความสำคัญ

&emsp;อาเซียนมีประชากรกว่า **650 ล้านคน** กระจายอยู่ใน 10 ประเทศ และกำลังก้าวเข้าสู่ยุคของการขยายตัวอย่างรวดเร็ว เมืองใหญ่ๆ เช่น กรุงเทพฯ จาการ์ตา และมะนิลา ต่างแบกรับประชากรหลักสิบล้านคน พร้อมกับความท้าทายด้านการจราจรและมลพิษที่ตามมา

&emsp;ระบบขนส่งสาธารณะจึงไม่ใช่แค่ "รถไฟฟ้า" หรือ "รถเมล์" — มันคือตัวชี้วัดว่าเมืองนั้น **พัฒนาไปถึงไหน** แล้ว

**ข้อมูลที่ใช้วิเคราะห์:**
- 📊 จำนวนผู้โดยสารรายเดือน จำแนกตามระบบขนส่ง (2019–2025)
- 💰 GDP รายเมือง (Billion USD)
- 👥 จำนวนประชากร (ล้านคน)
- 🌫️ ค่าฝุ่น PM2.5 (μg/m³)

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

---
## Part 5 — มะนิลา 🇵🇭
### ฟิลิปปินส์โตมา 25 ปีอย่างไร?

&emsp;มะนิลาพิเศษกว่าเมืองอื่นตรงที่ **มีข้อมูลตั้งแต่ปี 1999** ยาวนานที่สุดในชุดนี้ แต่กลับมีข้อมูลเพียงระบบเดียวคือ **Rail Network** (รวม LRT-1, LRT-2 และ MRT-3) ทำให้เราได้เห็นภาพการเดินทางของชาวมะนิลาตลอด 25 ปีเต็ม

&emsp;มะนิลามีปัญหาการจราจรที่ขึ้นชื่อว่าแย่ที่สุดในโลก ระบบ Rail จึงเป็น "ความหวัง" ของคนเมือง

---

In [ ]:
# ── Manila: prep data ─────────────────────────────────────────────────────────
mnl = df[df['City']=='Manila'].copy()
mnl_m   = mnl.groupby('Date')['Ridership'].sum().reset_index()
mnl_yr  = mnl.groupby(['Year','Mode'])['Ridership'].sum().reset_index()
mnl_yr['YoY'] = mnl_yr['Ridership'].pct_change()*100

# Focus 2015-2024 for recent view
mnl_recent = mnl[mnl['Year'].between(2015,2024)]
mnl_m_r = mnl_recent.groupby('Date')['Ridership'].sum().reset_index()
mnl_yr_r = mnl_recent.groupby('Year')['Ridership'].sum().reset_index()
mnl_yr_r['YoY'] = mnl_yr_r['Ridership'].pct_change()*100

print('Manila data ready ✅')

### ตารางที่ 5.1 — แนวโน้มผู้โดยสาร 25 ปี มะนิลา (1999–2024)

In [ ]:
# 5.1 Long-term area
mnl_long = mnl.groupby('Date')['Ridership'].sum().reset_index()
fig = px.area(mnl_long, x='Date', y='Ridership',
    title='<b>ตารางที่ 5.1</b>  มะนิลา — ผู้โดยสาร Rail Network ระยะ 25 ปี (1999–2024)',
    labels={'Ridership':'จำนวนผู้โดยสาร','Date':''},
    color_discrete_sequence=[CITY_COLORS['Manila']])
fig.update_traces(fillcolor='rgba(199,125,255,0.13)', line_color=CITY_COLORS['Manila'], line_width=2)
# Annotate COVID
fig.add_vrect(x0='2020-01-01', x1='2021-06-01', fillcolor='#FF4D4D', opacity=0.07,
              annotation_text='COVID-19', annotation_position='top left',
              annotation_font_color='#FF8080', annotation_font_size=10)
fig.update_layout(**base_layout(440), hovermode='x unified',
                  yaxis=dict(tickformat='.2s',**ax_style()), xaxis=ax_style())
fig.show()

# 5.2 Donut — only one mode so show yearly breakdown instead
# Use pie showing before/after COVID
mnl_era = pd.DataFrame({
    'Period': ['ก่อน COVID (1999–2019)','ระหว่าง COVID (2020–2021)','หลัง COVID (2022–2024)'],
    'Ridership': [
        mnl[mnl['Year'].between(1999,2019)]['Ridership'].sum(),
        mnl[mnl['Year'].between(2020,2021)]['Ridership'].sum(),
        mnl[mnl['Year'].between(2022,2024)]['Ridership'].sum(),
    ]
})
fig = px.pie(mnl_era, names='Period', values='Ridership',
    title='<b>ตารางที่ 5.2</b>  มะนิลา — สัดส่วนผู้โดยสารจำแนกตามยุค',
    hole=0.45, color_discrete_sequence=['#C77DFF','#FF6B6B','#A8E6CF'])
fig.update_traces(textposition='inside', textinfo='percent+label',
                  insidetextorientation='radial', textfont_size=11,
                  marker=dict(line=dict(color=PAPER_BG,width=2)))
fig.update_layout(**base_layout(440, legend_h=False),
                  legend=dict(bgcolor='rgba(22,27,34,0.9)',bordercolor=GRID_C,
                              borderwidth=1,font_size=11,orientation='h',y=-0.1))
fig.show()

# 5.3 Annual bar 2015-2024
mnl_yr_bar = mnl[(mnl['Year'].between(2015,2024))].groupby('Year')['Ridership'].sum().reset_index()
fig = px.bar(mnl_yr_bar, x='Year', y='Ridership',
    title='<b>ตารางที่ 5.3</b>  มะนิลา — ผู้โดยสารรายปี ช่วงล่าสุด (2015–2024)',
    labels={'Ridership':'ผู้โดยสาร','Year':''},
    color_discrete_sequence=[CITY_COLORS['Manila']])
fig.update_layout(**base_layout(440),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()), bargap=0.25)
fig.show()

# 5.4 YoY 2015-2024
mnl_yr_yoy = mnl_yr_bar.copy(); mnl_yr_yoy['YoY'] = mnl_yr_yoy['Ridership'].pct_change()*100
mnl_yr_yoy = mnl_yr_yoy.dropna()
fig = px.bar(mnl_yr_yoy, x='Year', y='YoY', text_auto='.0f',
    title='<b>ตารางที่ 5.4</b>  มะนิลา — อัตราการเปลี่ยนแปลง YoY (%) ช่วงล่าสุด (2015–2024)',
    labels={'YoY':'YoY Change (%)','Year':''},
    color_discrete_sequence=[CITY_COLORS['Manila']])
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(440),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()))
fig.update_traces(textfont_size=9, textposition='outside')
fig.show()

> 📌 **สรุป มะนิลา:** ภาพ 25 ปีบอกเราว่ามะนิลาเติบโตอย่างช้าๆ แต่มั่นคง จนกระทั่ง COVID ทำให้ตัวเลขดิ่งลง -67% ในปี 2020 ซึ่งหนักกว่าทุกเมืองในกลุ่ม และการฟื้นตัวก็ช้ากว่า — ในปี 2024 มะนิลายังไม่กลับมาถึงระดับ Pre-COVID สะท้อนว่าปัญหาขีดความสามารถของระบบ Rail ที่มีอยู่จำกัดกำลังฉุดรั้งการฟื้นตัว

---
## Part 6 — Economic Growth 💹
### ขนส่งสาธารณะกับเศรษฐกิจ — มีความสัมพันธ์กันจริงไหม?

&emsp;ลองคิดแบบนี้ครับ — เมื่อคนมีงานทำ มีเงินในกระเป๋า พวกเขาจะออกไปข้างนอกมากขึ้น ไปห้างสรรพสินค้า ไปทำงาน ไปพบเพื่อน ทุกการเดินทางนั้น ถ้าใช้ระบบขนส่งสาธารณะ มันจะปรากฏอยู่ในตัวเลข Ridership

&emsp;ในทางกลับกัน เมื่อเศรษฐกิจหดตัว คนก็อยู่บ้านมากขึ้น ตัวเลขก็ลดลง

&emsp;ส่วนนี้จะทดสอบความสัมพันธ์นี้ผ่านข้อมูลจริงจาก 5 เมือง

---